# Notebook 11 — Overnight full JEPA comparison

**Run All starts the real overnight experiment in detached tmux.** It continues independently of the browser, with an absolute **12-hour cap**, checkpoint recovery, an independent deadline watcher and **hourly Codex agent reviews**. The supervisor records progress every 30 seconds and can request a validated operational repair if the worker crashes or stalls.

The completed small pilot favored ViT over both dense kernel models. This larger, prospectively registered comparison tests more training and six fresh paired seeds; it does not assume the new encoder is better.

| Encoder | Interaction | Shared structure |
|---|---|---|
| ViT | Learned self-attention | Patch stem, four residual/MLP blocks, normalization, JEPA predictor |
| Spatial kernel | Smooth positive kernel on fixed image positions | Same stem/MLPs/norms/predictor |
| Learned kernel | Smooth kernel on separate learned content/position coordinates | Same stem/MLPs/norms/predictor |

All use 128px images, 8px patches, width 192 and the unchanged full JEPA predictor/loss. Common parameters start equal with independent storage. Spatial/learned kernels also share value/output/kernel initialization. Active counts differ and are reported. Both online and EMA target encoders use the selected architecture.

**18 training arms:** seeds 20261401–20261406 and all three encoders. Physical/effective batch is 128. After discarded timing calibration, select the largest complete schedule from **2,400 / 1,800 / 1,200 / 1,000 updates** that fits the original clock with recovery and evaluation reserve. The choice uses timing only; outcomes cannot change it. Each initial, update-350 and final EMA encoder receives the same fixed 20 epoch linear probe: **54 evaluations**.

Primary comparison: learned kernel minus ViT final development accuracy across six paired seeds. Report paired means, individual differences, 95% t intervals and an exact sign-test robustness check. Six seeds can support useful inference but cannot guarantee statistical significance. Reused development membership limits the claims.


## Kernel encoder mathematics

For normalized patch features $u_i$ and position $r_i$, learn low-dimensional coordinates $a_i=\phi_Q(u_i,r_i)$ and $b_i=\phi_K(u_i,r_i)$, and values $v_i=Vu_i$. The spatial control uses the same coordinate space without content displacement. A positive mixture of smooth radial kernels defines

$$m_i=\frac{\sum_{j\in C}K_\theta(a_i,b_j)v_j}{\sum_{j\in C}K_\theta(a_i,b_j)}.$$

Only valid visible context patches enter a context encoder. Each block applies an output projection, residual connection and pointwise MLP. Both online and EMA target encoders use the selected architecture; predictor, target normalization and loss remain unchanged.

FMM would later approximate these same sums. Dense evaluation isolates the learning hypothesis. A useful kernel still needs a separate error-controlled acceleration and scaling test before an FMM claim.

The implementation evaluates this positive-kernel ratio through log-sum-exp and row softmax, without an added epsilon; every sample must contain a valid source.

Shared vectorized validity/overlap guards preserve forward outputs, losses and all gradients bit for bit while removing repeated GPU host synchronizations. Ordered CPU preprocessing preserves exact pixels, mask metadata and RNG/cursor state. These are execution changes shared by every architecture, not a kernel-model advantage.


## Inputs, checkpoints and exact recovery

Inputs: the existing sealed Food101 10,100 training/probe-training and 2,020 development images. Official test remains reserved. No labels enter the self-supervised loss.

Artifacts: artifacts/notebook_11/manifold-long-001/. Final report and visual files: stages/learning/visual_report/report.html and notebooks/11_Overnight_Results_Review.ipynb. The visual review includes accuracy by seed/step, learning gains, training-loss curves, time/memory comparisons, PCA feature views and actual image nearest-neighbor grids. Feature plots are not pixel reconstructions.

Full-state checkpoints: checkpoints/manifold-long-001/. Save at epoch/stage boundaries, every 50 updates or 15 minutes, retaining the latest two valid resumable states and best. Model/teacher, optimizer/scheduler/scaler, all RNGs, sampler/data/mask cursor, global step, config and metrics are saved atomically and verified through manifests. One writer per run.

Before long training, actual four-step versus interrupted two-plus-two recovery must be bitwise equal for every architecture. Real teacher feature/probe integration must pass. The prior diagnostic device mismatch is fixed and regression-tested in this study.

**Resume:** fresh ahr-go kernel, Run All unchanged with the same RUN_ID before the original deadline. Active worker reconnects; valid states resume exactly. Completed training evidence is reused for evaluation recovery. Invalid manifests fail explicitly; neither the launcher nor hourly agent may reset the clock or change scientific endpoints.

**Worker:** fmm-manifold-long-manifold-long-001. Attach: tmux attach -t fmm-manifold-long-manifold-long-001. Detach: Ctrl-b, then d. Independent deadline and heartbeat sessions have the same run ID. GPU 0 is used; unrelated AHR work on GPU 1 is preserved.

**Hourly agent:** reviews checkpoint progress and errors. For an operational failure it may create a separately bound notebook adapter, validate it and resume the same frozen study. It cannot change models, data, seeds, optimization, quality selection or the original clock. Agent review logs: heartbeat/agent_reviews/; latest status: heartbeat/latest_heartbeat.json.

Workspace is on persistent boot disk; instance-deletion policy is unverified. Copy final/best checkpoints off-instance and verify hashes before recycling/deleting the instance.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import nbformat, importlib.util, sys
from IPython.display import display, Markdown, Image
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm11_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/11_manifold_long/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Registered configuration
Six fresh paired seeds, three architectures, timing-only schedule candidates, intermediate/final evaluation, source-bound execution improvements and the 12-hour cap are fixed before outcomes.

In [ ]:
RUN_ID = "manifold-long-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG


## 3. Readiness
Read-only check of operator, input, execution, pipeline, visual, supervisor and live-agent certificates. It creates no experiment clock or worker.

In [ ]:
READINESS = WORKFLOW.readiness(ROOT, CONFIG)
import pandas as pd
display(pd.DataFrame(READINESS["architecture_summary"]))
display({key: READINESS[key] for key in ("status", "production_run_started", "expected_runtime", "planned_training_runs", "planned_probe_evaluations", "checkpoint_location", "scientific_results")})


## 4. Launch or reconnect
This cell starts the real overnight study. All preflights, training, evaluations and visual report generation run independently in tmux under the original cap.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
DISPLAY_LAUNCH = {key: value for key, value in LAUNCH.items() if key != "command"}
if LAUNCH.get("deadline_unix"):
    DISPLAY_LAUNCH["hard_stop_dublin"] = datetime.fromtimestamp(LAUNCH["deadline_unix"], ZoneInfo("Europe/Dublin")).isoformat()
display(DISPLAY_LAUNCH)


## 5. Inspect progress and heartbeat
Rerun this cell for checkpoint counts and the latest server-side heartbeat.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
PROGRESS = STATUS.get("saved_training_progress", [])
display({key: value for key, value in STATUS.items() if key != "saved_training_progress"})
if PROGRESS:
    display(pd.DataFrame(PROGRESS))

HEARTBEAT=ROOT/'artifacts/notebook_11'/RUN_ID/'heartbeat/latest_heartbeat.json'
if HEARTBEAT.exists():
    import json
    LIVE_HEARTBEAT=json.loads(HEARTBEAT.read_text())
    display({k:LIVE_HEARTBEAT.get(k) for k in ('observed_unix','worker_alive','activity_age_seconds','remaining_seconds','gpu_csv','evaluation_progress')})
PLAN_PATH=ROOT/'artifacts/notebook_11'/RUN_ID/'stages/learning/plan.json'
if PLAN_PATH.exists():
    PLAN=json.loads(PLAN_PATH.read_text())
    display({'selected_final_updates':PLAN['updates'],'milestones':PLAN['milestones'],'budget_selection':PLAN['budget_selection']})


## 6. Read completed results
The separate read-only results notebook and embedded HTML report give the complete visual review.

In [ ]:

REPORT_FOLDER=ROOT/'artifacts/notebook_11'/RUN_ID/'stages/learning'
REPORT=REPORT_FOLDER/'stage_results.md'
if STATUS.get('status')=='COMPLETED' and REPORT.exists():
    display(Markdown('\n'.join(line for line in REPORT.read_text().splitlines() if not line.startswith('!['))))
    display(Markdown('Open **notebooks/11_Overnight_Results_Review.ipynb** for the image examples and full visual report.'))
    for figure in ('stage_quality.png','stage_compute.png','stage_contrasts.png'):
        p=REPORT_FOLDER/figure
        if p.exists():display(Image(filename=str(p)))
else:
    display(Markdown('**Study is still running or incomplete.** Open the read-only results notebook to inspect progress. Missing comparisons remain unknown; partial scores do not establish a winner.'))


## Interpretation

Read the primary learned-minus-ViT final differences across all six seeds. Use own-initialization gains and the intermediate350 snapshot to understand learning, and the fixed spatial kernel to test the contribution of content geometry. Inspect measured complete-step cost and memory rather than inferring hardware speed from parameter counts.

Kernel sums are computed densely. This tests representation learning; an error-controlled FMM approximation and high-token scaling study remain separate work. Learned coordinates are not proved manifold charts. Longer training, more seeds or lower JEPA loss alone do not guarantee better semantic representations.

References: [I-JEPA](https://arxiv.org/abs/2301.08243), [Fast Multipole Attention](https://arxiv.org/abs/2310.11960), [non-interactive Codex for scheduled jobs](https://learn.chatgpt.com/docs/non-interactive-mode). This is a controlled custom research implementation.
